In [5]:
#!pip install llama-index-embeddings-huggingface
#!pip install -U kaleido
#!pip install llama-index-llms-google-genai
#!pip install --upgrade llama-index-core llama-index-llms-google-genai
#!pip install nest-asyncio
#!pip install llama-index-llms-ollama
#!pip install llama-index-llms-groq

In [2]:
def initialize_llm(provider: str = "openai", temperature: float = 0.3):
    """
    Initialize and return LLM instance based on provider.
    """
    load_dotenv(override=True)
    
    if provider.lower() == "openai":
        from llama_index.llms.openai import OpenAI
        api_key = os.getenv("OPENAI_API_KEY")
        return OpenAI(model="gpt-4o-mini", temperature=temperature, api_key=api_key)
    
    elif provider.lower() == "gemini":
        from llama_index.llms.gemini import Gemini
        api_key = os.getenv("GEMINI_API_KEY")
        return Gemini(model="gemini-1.5-flash", api_key=api_key, temperature=temperature)
    
    elif provider.lower() == "ollama":
        from llama_index.llms.ollama import Ollama
        return Ollama(model="llama3.2", temperature=temperature, request_timeout=120.0)
    
    elif provider.lower() == "groq":
        from llama_index.llms.groq import Groq
        api_key = os.getenv("GROQ_API_KEY")
        if not api_key:
            raise EnvironmentError("GROQ_API_KEY not found in environment variables")
        # llama-3.3-70b-versatile is currently one of their best/fastest models
        return Groq(model="llama-3.3-70b-versatile", api_key=api_key, temperature=temperature)
    
    else:
        raise ValueError(f"Unknown provider: {provider}. Choose 'openai', 'gemini', 'ollama', or 'groq'")

In [ ]:
import os
import csv
import io
import pandas as pd
import gradio as gr
from dotenv import load_dotenv
from sqlalchemy import create_engine, inspect
from llama_index.core import SQLDatabase, VectorStoreIndex
from llama_index.core.indices.struct_store.sql_query import SQLTableRetrieverQueryEngine
from llama_index.core.objects import SQLTableNodeMapping, SQLTableSchema, ObjectIndex
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import plotly.express as px
import plotly.io as pio
import tempfile
from typing import Tuple, Dict, List
import ast
import json

# 🔧 Constants
EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
LLM_PROVIDER = "groq"  # Change to "gemini" to use Gemini, "ollama" for Ollama, or "openai" for OpenAI
input_folder = r"G:\My Drive\python\llm\udemy_llm\llm_engineering\cte\knowledge-base"
csv_path = os.path.join(input_folder, "aact_multiple_sponsors.csv")

# Initialize chat history
chat_history = []

# 🌿 Load environment variables
load_dotenv(override=True)
hf_token = os.getenv("HUGGINGFACE_API_KEY")

os.environ["HF_TOKEN"] = hf_token



# 🧠 Create SQL engine and table
engine = create_engine("sqlite:///trial_data.db")

# Only load CSV and write to DB if the file doesn't exist yet
if not os.path.exists("trial_data.db"):
    print("🚀 Database not found. Initializing from CSV...")

    # 📊 Load CSV
    df1 = pd.read_csv(
        csv_path,
        encoding="utf-8",
        sep="|",
        quoting=csv.QUOTE_MINIMAL,
        quotechar='"',
        on_bad_lines='skip',
        low_memory=False
    )
    df1.to_sql("aact_multiple_sponsors", engine, index=False, if_exists="replace")
    print("✅ Database created successfully.")
else:
    print("📂 Using existing final database (skipping CSV load).")

def get_table_schema_info(engine, table_name: str) -> Dict:
    """Get detailed table schema information including column names, types, and sample data"""
    inspector = inspect(engine)
    columns = inspector.get_columns(table_name)
    
    # Get sample data for context
    sample_query = f"SELECT * FROM {table_name} LIMIT 5"
    sample_df = pd.read_sql(sample_query, engine)
    
    schema_info = {
        "table_name": table_name,
        "columns": [{"name": col["name"], "type": str(col["type"])} for col in columns],
        "column_names": [col["name"] for col in columns],
        "sample_data": sample_df.to_dict('records'),
        "total_rows": pd.read_sql(f"SELECT COUNT(*) as count FROM {table_name}", engine).iloc[0]['count']
    }
    
    return schema_info

# Get schema information
schema_info = get_table_schema_info(engine, "aact_multiple_sponsors")

# Enhanced context with semantic mappings
enhanced_context = f"""
This table contains clinical trial data with the following structure:

Table: aact_multiple_sponsors
Total Records: {schema_info['total_rows']}

Columns and their meanings:
{json.dumps(schema_info['columns'], indent=2)}

Sample data:
{json.dumps(schema_info['sample_data'][:3], indent=2)}

SEMANTIC MAPPINGS - Use these to understand user intent:
- "clinical trials", "studies", "trials" → refer to rows in the table
- "enrollment", "participants", "subjects", "patients" → enrollment column
- "study id", "trial id", "study identifier" → nct_id or studyid columns
- "phase", "trial phase", "study phase" → phase column
- "sponsor", "funding", "organization" → sponsor_name column
- "condition", "disease", "indication" → conditions column
- "title", "study title", "trial name" → brief_title or official_title columns
- "status", "trial status" → overall_status column
- "start date", "beginning" → start_date column
- "completion", "end date" → completion_date column

IMPORTANT: Always use exact column names from the schema when generating SQL queries.
"""

# 🪄 Initialize LLM and query engine
try:
    #llm = OpenAI(model=LLM_MODEL_ID, temperature=0.3)  # Lower temperature for more consistent SQL
    llm = initialize_llm(provider=LLM_PROVIDER, temperature=0.3)
    sql_database = SQLDatabase(engine, include_tables=["aact_multiple_sponsors"])
    
    # Create embedding model
    embed_model = HuggingFaceEmbedding(model_name=EMBEDDING_MODEL_ID)
    
    # Create table schema objects with enhanced context
    table_node_mapping = SQLTableNodeMapping(sql_database)
    table_schema_objs = [
        SQLTableSchema(table_name="aact_multiple_sponsors", context_str=enhanced_context)
    ]
    
    # Create object index with proper parameters
    obj_index = ObjectIndex.from_objects(
        table_schema_objs,
        table_node_mapping,
        VectorStoreIndex,
        embed_model=embed_model
    )
    
    # Create query engine
    query_engine = SQLTableRetrieverQueryEngine(
        sql_database,
        obj_index.as_retriever(similarity_top_k=1),
        llm=llm
    )
    
    print("✅ Successfully initialized LlamaIndex components")
    
except Exception as e:
    print(f"❌ Error initializing LlamaIndex components: {e}")
    #llm = OpenAI(model=LLM_MODEL_ID, temperature=0.3)
    llm = initialize_llm(provider=LLM_PROVIDER, temperature=0.3)
    sql_database = SQLDatabase(engine, include_tables=["aact_multiple_sponsors"])
    query_engine = None

def run_sql_query(query: str, sql_database) -> pd.DataFrame:
    """Enhanced SQL query execution with better error handling"""
    try:
        result = sql_database.run_sql(query)
        
        if isinstance(result, str):
            try:
                parsed_result = ast.literal_eval(result)
                if isinstance(parsed_result, tuple) and all(isinstance(item, tuple) for item in parsed_result):
                    result = list(parsed_result)
                elif isinstance(parsed_result, list) and all(isinstance(item, tuple) for item in parsed_result):
                    result = parsed_result
            except (ValueError, SyntaxError):
                pass

        if isinstance(result, tuple):
            result = result[1]

        if hasattr(result, 'fetchall') and hasattr(result, 'keys'):
            data = result.fetchall()
            columns = result.keys()
            df = pd.DataFrame(data, columns=columns)
        elif isinstance(result, list) and all(isinstance(item, tuple) for item in result):
            if result:
                num_columns = len(result[0])
                generated_columns = [f'Column_{i}' for i in range(num_columns)]
                df = pd.DataFrame(result, columns=generated_columns)
            else:
                df = pd.DataFrame()
        elif isinstance(result, dict) and 'result' in result and 'col_keys' in result:
            df = pd.DataFrame(result['result'], columns=result['col_keys'])
        elif isinstance(result, str):
            df = pd.DataFrame([{'Status': result}])
        else:
            df = pd.DataFrame()

        return df
    except Exception as e:
        print(f"SQL execution error: {e}")
        return pd.DataFrame([{'Error': str(e)}])

def intelligent_sql_generation(user_question: str, schema_info: Dict) -> str:
    """Generate SQL with better context awareness and error handling"""
    
    col_details = ', '.join([f"{col['name']} ({col['type']})" for col in schema_info['columns']])
    prompt = f"""You are an expert SQL generator for clinical trial data. 

TASK: Convert the user question into a precise SQL query.

USER QUESTION: "{user_question}"

DATABASE SCHEMA:
Table: {schema_info['table_name']}
Available columns: {col_details}

SEMANTIC UNDERSTANDING RULES:
1. "clinical trials", "studies", "trials" → refer to rows in the table
2. "enrollment", "participants", "subjects" → use 'enrollment' column
3. "study id", "trial id" → use 'nct_id' or 'studyid' columns (prefer nct_id if available)
4. "phase" → use 'phase' column
5. "sponsor" → use 'sponsor_name' column
6. "condition", "disease" → use 'conditions' column
7. "title", "study title" → use 'brief_title' or 'official_title' columns
8. "status" → use 'overall_status' column

REQUIREMENTS:
- Use ONLY column names that exist in the schema
- For "top N" queries, use ORDER BY with LIMIT
- Handle NULL values appropriately
- Use proper SQL syntax for SQLite
- Be specific about what to select and order by

EXAMPLE MAPPINGS:
- "top 10 studies by enrollment" → SELECT nct_id, brief_title, enrollment FROM aact_multiple_sponsors ORDER BY enrollment DESC LIMIT 10
- "studies by phase" → SELECT phase, COUNT(*) as study_count FROM aact_multiple_sponsors GROUP BY phase

Generate only the SQL query without any explanation or formatting:"""

    try:
        sql_response = llm.complete(prompt)
        sql_query = sql_response.text.strip()
        
        # Clean up SQL query
        if sql_query.startswith("```sql"):
            sql_query = sql_query.replace("```sql", "").replace("```", "").strip()
        elif sql_query.startswith("```"):
            sql_query = sql_query.replace("```", "").strip()
        
        # Validate that the query uses valid column names
        available_columns = set(schema_info['column_names'])
        used_columns = set()
        
        # Simple column extraction (this could be made more sophisticated)
        for col in available_columns:
            if col.lower() in sql_query.lower():
                used_columns.add(col)
        
        return sql_query
        
    except Exception as e:
        return f"SELECT 'Error generating SQL: {e}' as error_message"

def autonomous_chart_selection(df: pd.DataFrame, user_question: str) -> Tuple[str, str, str]:
    """Use LLM to autonomously select the best chart type based on data and user intent"""
    
    # Analyze the dataframe structure
    df_info = {
        "shape": df.shape,
        "columns": list(df.columns),
        "dtypes": {col: str(dtype) for col, dtype in df.dtypes.items()},
        "sample_data": df.head(3).to_dict('records') if not df.empty else [],
        "numeric_columns": df.select_dtypes(include=['number']).columns.tolist(),
        "categorical_columns": df.select_dtypes(include=['object']).columns.tolist()
    }
    
    prompt = f"""You are a data visualization expert. Analyze this data and select the BEST chart type.

USER QUESTION: "{user_question}"

DATA ANALYSIS:
- Shape: {df_info['shape']} (rows, columns)
- Columns: {df_info['columns']}
- Data types: {df_info['dtypes']}
- Numeric columns: {df_info['numeric_columns']}
- Categorical columns: {df_info['categorical_columns']}
- Sample data: {df_info['sample_data']}

AVAILABLE CHART TYPES:
1. "bar" - For categorical data with numeric values, comparisons
2. "line" - For time series, trends over time
3. "pie" - For parts of a whole, percentages (max 10 categories)
4. "scatter" - For correlations between two numeric variables
5. "histogram" - For distribution of single numeric variable
6. "box" - For showing statistical distribution
7. "table" - When visualization isn't appropriate

SELECTION CRITERIA:
- Consider the user's intent and question type
- Look at data structure and relationships
- Choose the most informative visualization
- Avoid pie charts for >10 categories
- Use tables only when charts don't add value

Respond in this EXACT format:
Chart Type: <type>
X-axis: <column name for x-axis, or 'N/A'>
Y-axis: <column name for y-axis, or 'N/A'>"""

    try:
        response = llm.complete(prompt)
        output = response.text.strip()
        
        # Parse the response
        chart_type = "table"
        x_axis = "N/A"
        y_axis = "N/A"
        
        for line in output.split('\n'):
            line = line.strip()
            if line.startswith("Chart Type:"):
                chart_type = line.split("Chart Type:")[1].strip().lower()
            elif line.startswith("X-axis:"):
                x_axis = line.split("X-axis:")[1].strip()
            elif line.startswith("Y-axis:"):
                y_axis = line.split("Y-axis:")[1].strip()
        
        return chart_type, x_axis, y_axis
        
    except Exception as e:
        print(f"Error in autonomous chart selection: {e}")
        # Fallback to simple heuristics
        if len(df.columns) == 2 and pd.api.types.is_numeric_dtype(df.iloc[:, 1]):
            return "bar", df.columns[0], df.columns[1]
        else:
            return "table", "N/A", "N/A"

def generate_enhanced_plot(df: pd.DataFrame, chart_type: str, x_axis: str, y_axis: str):
    """Generate plot with enhanced chart types, colors, and better error handling"""
    try:
        if df.empty or chart_type == 'table':
            return None
        
        # 1. Resolve effective x and y axes to avoid repetitive fallback logic
        eff_x = x_axis if x_axis != 'N/A' and x_axis in df.columns else (df.columns[0] if len(df.columns) > 0 else None)
        eff_y = y_axis if y_axis != 'N/A' and y_axis in df.columns else (df.columns[1] if len(df.columns) > 1 else eff_x)
        
        if not eff_x:
            return None
        
        # 2. Determine color column smartly
        categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
        color_col = None
        if categorical_cols:
            valid_color_cols = [c for c in categorical_cols if c != eff_x and c != eff_y]
            if valid_color_cols:
                color_col = valid_color_cols[0]
            elif eff_x in categorical_cols:
                color_col = eff_x
        
        custom_colors = px.colors.qualitative.Pastel
        kwargs = {"color_discrete_sequence": custom_colors}
        if color_col:
            kwargs["color"] = color_col
            
        fig = None
        
        # 3. Generate specific charts without redundant checks
        if chart_type == "bar":
            fig = px.bar(df, x=eff_x, y=eff_y, title=f"{eff_y} by {eff_x}", **kwargs)
        elif chart_type == "line":
            fig = px.line(df, x=eff_x, y=eff_y, title=f"{eff_y} Trend over {eff_x}", **kwargs)
        elif chart_type == "pie":
            df_pie = df.nlargest(10, eff_y) if len(df) > 10 and pd.api.types.is_numeric_dtype(df[eff_y]) else df
            fig = px.pie(df_pie, names=eff_x, values=eff_y, title=f"Distribution of {eff_y} by {eff_x}", color_discrete_sequence=custom_colors)
        elif chart_type == "scatter":
            num_cols = df.select_dtypes(include=['number']).columns.tolist()
            if len(num_cols) >= 2:
                sx = eff_x if eff_x in num_cols else num_cols[0]
                sy = eff_y if eff_y in num_cols else (num_cols[1] if sx == num_cols[0] else num_cols[0])
                fig = px.scatter(df, x=sx, y=sy, title=f"{sy} vs {sx}", **kwargs)
        elif chart_type == "histogram":
            hx = eff_x if pd.api.types.is_numeric_dtype(df[eff_x]) else (df.select_dtypes(include=['number']).columns[0] if not df.select_dtypes(include=['number']).empty else eff_x)
            fig = px.histogram(df, x=hx, title=f"Distribution of {hx}", **kwargs)
        elif chart_type == "box":
            by = eff_y if pd.api.types.is_numeric_dtype(df[eff_y]) else (df.select_dtypes(include=['number']).columns[0] if not df.select_dtypes(include=['number']).empty else None)
            if by:
                fig = px.box(df, y=by, title=f"Box Plot of {by}", **kwargs)
        
        if not fig:
            return None
            
        # Enhance styling with modern, cleaner looks
        fig.update_layout(
            template="plotly_white",
            font=dict(size=13),
            title_font=dict(size=18),
            plot_bgcolor='rgba(0,0,0,0)',
            paper_bgcolor='rgba(0,0,0,0)',
            margin=dict(l=40, r=40, t=60, b=40),
            hoverlabel=dict(bgcolor="white", font_size=12),
            showlegend=True if color_col or chart_type == 'pie' else False,
        )
        
        # Subtle gridlines and styling
        fig.update_xaxes(showgrid=False, zeroline=False)
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(211, 211, 211, 0.4)', zeroline=False)
        fig.update_traces(marker=dict(line=dict(width=0.5, color='rgba(0,0,0,0.2)')))
        
        return fig
        
    except Exception as e:
        print(f"Error generating plot: {e}")
        return None

def generate_data_insights(df: pd.DataFrame, user_question: str, sql_query: str) -> str:
    """Generate LLM insights about the retrieved data"""
    
    if df.empty:
        return "No data was retrieved for analysis."
    
    # Prepare data summary
    data_summary = {
        "total_rows": len(df),
        "columns": list(df.columns),
        "sample_data": df.head(5).to_dict('records'),
        "basic_stats": {}
    }
    
    # Add basic statistics for numeric columns
    numeric_cols = df.select_dtypes(include=['number']).columns
    for col in numeric_cols:
        if not df[col].empty:
            data_summary["basic_stats"][col] = {
                "mean": df[col].mean(),
                "median": df[col].median(),
                "min": df[col].min(),
                "max": df[col].max(),
                "std": df[col].std()
            }
    
    prompt = f"""You are a data analyst. Provide insights about the retrieved data based on the user's question.

USER QUESTION: "{user_question}"
SQL QUERY EXECUTED: {sql_query}

DATA RETRIEVED:
- Total rows: {data_summary['total_rows']}
- Columns: {data_summary['columns']}
- Sample data: {data_summary['sample_data']}
- Statistics: {data_summary['basic_stats']}

PROVIDE:
1. Answer to the user's specific question in a tabular format
2. Key findings from the data or Notable patterns or insights 
3. Any limitations or caveats

Keep your response concise but informative (1-2 sentences)."""

    try:
        response = llm.complete(prompt)
        return response.text.strip()
    except Exception as e:
        return f"Unable to generate insights due to error: {e}"

def autonomous_response(user_question: str, query_engine, sql_db):
    """Enhanced autonomous response with better context awareness and insights"""
    
    # Step 1: Generate contextually aware SQL
    try:
        sql_query = intelligent_sql_generation(user_question, schema_info)
        print(f"Generated SQL: {sql_query}")
        
        if "error" in sql_query.lower():
            return sql_query, None, None
            
    except Exception as e:
        return f"❌ Error generating SQL query: {e}", None, None
    
    # Step 2: Execute SQL
    try:
        result_df = run_sql_query(sql_query, sql_db)
        if result_df.empty:
            return "❌ No data found for your query.", None, None
        
        print(f"Retrieved {len(result_df)} rows")
        
    except Exception as e:
        return f"❌ SQL execution error: {e}", None, None
    
    import concurrent.futures
    
    # Step 3 & 4: Generate data insights and chart selection concurrently
    insights = ""
    chart_type, x_axis, y_axis = "table", "N/A", "N/A"
    
    with concurrent.futures.ThreadPoolExecutor() as executor:
        future_insights = executor.submit(generate_data_insights, result_df, user_question, sql_query)
        future_chart = executor.submit(autonomous_chart_selection, result_df, user_question)
        
        try:
            insights = future_insights.result()
        except Exception as e:
            insights = f"Could not generate insights: {e}"
        
        try:
            chart_type, x_axis, y_axis = future_chart.result()
        except Exception as e:
            print(f"Chart selection error: {e}")
    
    # Step 5: Generate visualization
    try:
        if chart_type != "table":
            plot_fig = generate_enhanced_plot(result_df, chart_type, x_axis, y_axis)
        else:
            plot_fig = None
    except Exception as e:
        print(f"Visualization error: {e}")
        plot_fig = None
    
    # Step 6: Compose comprehensive response
    response_text = f"""✅ **Query Executed:**
```sql
{sql_query}
```

📊 **Data Retrieved:** {len(result_df)} rows

🔍 **Data Insights:**
{insights}

📈 **Visualization:** {chart_type.title()} chart generated
"""
    
    if plot_fig is None and chart_type != "table":
        response_text += f"\n⚠️ Could not generate {chart_type} chart. Showing data table instead."
        # Add sample data to response
        if len(result_df) <= 10:
            response_text += f"\n\n**Complete Data:**\n```\n{result_df.to_string(index=False)}\n```"
        else:
            response_text += f"\n\n**Sample Data (first 10 rows):**\n```\n{result_df.head(10).to_string(index=False)}\n```"
    
    return response_text, plot_fig, result_df

# Gradio interface setup (rest of the code remains similar but with enhanced functionality)
def create_gradio_interface():
    """Create enhanced Gradio interface"""
    with gr.Blocks(title="Enhanced Clinical Trial Data Assistant", theme=gr.themes.Soft()) as demo:
        gr.Markdown("# 🏥 Enhanced Clinical Trial Data Assistant")
        gr.Markdown("Ask questions about clinical trial data and get intelligent visualized insights with AI-powered analysis!")
        
        # Add example questions
        gr.Markdown("""
        **Example Questions:**
        - "Show me the top 10 clinical trials by enrollment"
        - "What is the distribution of studies by phase?"
        - "Which sponsors have the most trials?"
        - "Show me trials for diabetes conditions"
        """)
        
        with gr.Row():
            with gr.Column(scale=2):
                chatbot = gr.Chatbot(
                    value=[],
                    height=500,
                    label="Chat History",
                    show_label=True
                )
                msg = gr.Textbox(
                    placeholder="Ask about clinical trials (e.g., 'Show me trials by phase')",
                    label="Your Question",
                    lines=2
                )
                with gr.Row():
                    submit_btn = gr.Button("Submit", variant="primary", scale=2)
                    clear_btn = gr.Button("Clear Chat", variant="secondary", scale=1)
            
            with gr.Column(scale=1):
                plot_output = gr.Plot(label="", visible=True)
        
        # Event handlers
        def submit_and_clear(user_question, history):
            if not user_question.strip():
                return history, "", None
            
            try:
                response_text, plot_fig, result_df = autonomous_response(
                    user_question, query_engine, sql_database
                )
                new_history = history + [[user_question, response_text]]
                return new_history, "", plot_fig
            except Exception as e:
                error_message = f"❌ Error processing your request: {str(e)}"
                new_history = history + [[user_question, error_message]]
                return new_history, "", None
        
        def clear_chat():
            return [], None
        
        # Connect events
        submit_btn.click(
            submit_and_clear,
            inputs=[msg, chatbot],
            outputs=[chatbot, msg, plot_output]
        )
        
        msg.submit(
            submit_and_clear,
            inputs=[msg, chatbot],
            outputs=[chatbot, msg, plot_output]
        )
        
        clear_btn.click(
            clear_chat,
            outputs=[chatbot, plot_output]
        )
    
    return demo

def test_enhanced_system():
    """Test the enhanced system"""
    print("🧪 Testing enhanced system...")
    try:
        # Test schema info
        print(f"✅ Schema loaded: {len(schema_info['columns'])} columns")
        
        # Test intelligent SQL generation
        test_sql = intelligent_sql_generation("Show me top 5 studies by enrollment", schema_info)
        print(f"✅ SQL generation test: {test_sql[:100]}...")
        
        # Test full pipeline
        response_text, plot_fig, result_df = autonomous_response(
            "What is study count by phase?", 
            query_engine, 
            sql_database
        )
        print(f"✅ Full pipeline test successful")
        
        return True
    except Exception as e:
        print(f"❌ Enhanced system test failed: {e}")
        return False

# Launch the enhanced interface
if __name__ == "__main__":
    if not test_enhanced_system():
        print("❌ Enhanced system test failed. Please check your setup.")
        exit(1)
    
    try:
        demo = create_gradio_interface()
        demo.launch(server_name="127.0.0.1")
    except Exception as e:
        print(f"Error launching Gradio interface: {e}")
        print("Make sure all dependencies are installed and API keys are configured.")

📂 Using existing final database (skipping CSV load).
✅ Successfully initialized LlamaIndex components
🧪 Testing enhanced system...
✅ Schema loaded: 41 columns
✅ SQL generation test: SELECT nct_id, brief_title, enrollment FROM aact_multiple_sponsors ORDER BY enrollment DESC LIMIT 5...
Generated SQL: SELECT study_phase, COUNT(*) as study_count FROM aact_multiple_sponsors GROUP BY study_phase
Retrieved 8 rows
✅ Full pipeline test successful


C:\Users\nitin\AppData\Local\Temp\ipykernel_50344\1079118943.py:549: UserWarning:

You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.



* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Generated SQL: SELECT nct_id, brief_title, enrollment FROM aact_multiple_sponsors WHERE sponsor_name = 'Novartis' ORDER BY enrollment DESC LIMIT 5
Retrieved 5 rows
